# Introduction to financial technical analysis
### Course Project — Part 1, Section B: Golden / Death Crosses on the S&P 500

**Data window:** 1 year of daily prices (2024-05-01 → 2025-05-01)
**Universe:** every constituent of the S&P 500 index
**Signals:** MA50 / MA200 crossovers (Golden Cross and Death Cross) detected in the last 14 days

# 📌 Objectives

By the end of this notebook, students will be able to:

1. **Access and Manipulate Financial Data:**
   - Use the `yfinance` library to retrieve historical stock prices for all companies in the S&P 500 index.

2. **Compute Key Technical Indicators:**
   - Calculate 50-day and 200-day moving averages for each stock in the index.

3. **Detect Trading Signals:**
   - Identify Golden Crosses and Death Crosses as described in technical analysis literature.

4. **Analyze Recent Market Behavior:**
   - Isolate and examine stocks that experienced technical signals (crosses) in the last 14 days.

5. **Visualize Price Trends and Volatility:**
   - Plot historical price movements along with moving averages, and compute volatility around signal dates.

6. **Interpret Technical Indicators in Context:**
   - Reflect on what Golden and Death Crosses signify and how traders may respond to them.

7. **Evaluate Strategy Viability:**
   - Discuss the strengths and limitations of using moving averages as a standalone trading strategy.

8. **Connect Technical Analysis to Broader Market Intelligence:**
   - Explore how sentiment analysis and news (covered in later sections) could complement technical signals.

9. **Develop Critical Thinking About Signal Reliability:**
   - Assess potential risks of false positives and propose improvements or filters to enhance signal accuracy.

10. **Engage in Strategic Reflection:**
    - Answer analytical questions aimed at understanding the utility, risks, and presentation of the strategy to a professional audience.

## Import and install librairies

In [ ]:
%pip install --quiet pandas numpy yfinance lxml matplotlib

In [ ]:
import time
import warnings

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np
import pandas as pd
import yfinance as yf
from IPython.display import Markdown, display

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

print("pandas", pd.__version__, "| numpy", np.__version__, "| yfinance", yf.__version__)

## Get the list of stocks in the S&P 500 

In [ ]:
# Read and print the stock tickers that make up S&P500
df_tickers = pd.read_html(
    'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies')[0]
print(df_tickers.head())

In [ ]:
display(df_tickers)

In [ ]:
ticker_list = df_tickers['Symbol'].tolist()

### Ticker hygiene before hitting the API

Wikipedia writes class shares with a dot (`BRK.B`, `BF.B`), while Yahoo Finance expects a dash
(`BRK-B`, `BF-B`). If we send the raw symbols, those tickers come back empty and we silently lose
stocks from the universe, so we build an explicit *Wikipedia symbol → Yahoo symbol* mapping and keep
it around to translate results back at the end.

In [ ]:
# Yahoo Finance uses '-' instead of '.' for share classes (BRK.B -> BRK-B)
yahoo_symbols = [t.strip().replace('.', '-') for t in ticker_list]
yahoo_to_wiki = dict(zip(yahoo_symbols, ticker_list))

# Sector / company metadata, useful to contextualize the signals later on
meta = (df_tickers.assign(YAHOO_SYMBOL=yahoo_symbols)
                  .set_index('YAHOO_SYMBOL')[['Security', 'GICS Sector']]
                  .rename(columns={'Security': 'COMPANY', 'GICS Sector': 'SECTOR'}))

print(f"S&P 500 constituents retrieved : {len(ticker_list)}")
print("Symbols renamed for Yahoo       :",
      [f"{w} -> {y}" for w, y in zip(ticker_list, yahoo_symbols) if w != y])
display(meta.head())

## Get the closing price of all 500 stocks in the S&P 500 Index
Use the yfinance library to retrieve the close price of all 500 stocks in the index between 2024-05-01 and 2025-05-01
https://ranaroussi.github.io/yfinance/reference/yfinance.stock.html

In [ ]:
start_date = '2024-05-01'
end_date = '2025-05-01'

### Why we download more than one year of history

The analysis window required by the assignment is **2024-05-01 → 2025-05-01** (~251 trading days).
A 200-day moving average needs 200 observations *before* it produces its first value, so if we only
downloaded the analysis window the MA200 would be undefined for the first ~200 days and we could only
study crosses in the last ~50 sessions.

To compute **MA50 and MA200 correctly over the whole analysis window** we download a *warm-up* period
of ~420 extra calendar days (~290 trading days) and then slice back to the required window once the
indicators are computed. `df_close` — the deliverable of this section — still covers exactly
2024-05-01 → 2025-05-01.

In [ ]:
warmup_start = (pd.Timestamp(start_date) - pd.Timedelta(days=420)).strftime('%Y-%m-%d')
print(f"Download window : {warmup_start} -> {end_date}  (warm-up included)")
print(f"Analysis window : {start_date} -> {end_date}")

In [ ]:
def download_close_prices(symbols, start, end, batch_size=60, pause=1.0, retries=2):
    """Download adjusted closing prices for a list of tickers, in batches.

    Batching keeps every request small enough for the Yahoo endpoint (a single call with 500+
    symbols is frequently throttled) and lets us retry only the batches that failed.
    Returns a (dates x tickers) DataFrame of closing prices.
    """
    frames, missing = [], []

    for i in range(0, len(symbols), batch_size):
        batch = symbols[i:i + batch_size]
        data = None
        for attempt in range(retries + 1):
            try:
                data = yf.download(batch, start=start, end=end, auto_adjust=True,
                                   progress=False, group_by='column', threads=True)
                if data is not None and not data.empty:
                    break
            except Exception as exc:                      # network hiccup / throttling
                print(f"  batch {i // batch_size + 1} attempt {attempt + 1} failed: {exc}")
            time.sleep(pause * (attempt + 1))

        if data is None or data.empty:
            missing.extend(batch)
            continue

        # A single-ticker batch comes back with flat columns; a multi-ticker batch is a MultiIndex
        if isinstance(data.columns, pd.MultiIndex):
            close = data['Close'].copy()
        else:
            close = data[['Close']].copy()
            close.columns = batch[:1]

        frames.append(close)
        print(f"  batch {i // batch_size + 1:>2}: {len(batch):>3} tickers -> "
              f"{close.shape[1]} series, {close.shape[0]} rows")
        time.sleep(pause)

    df = pd.concat(frames, axis=1).sort_index()
    df = df.loc[:, ~df.columns.duplicated()]
    return df, missing


df_close_full, failed_batches = download_close_prices(yahoo_symbols, warmup_start, end_date)
print(f"\nRaw download: {df_close_full.shape[0]} dates x {df_close_full.shape[1]} tickers")

In [ ]:
# --- Data quality control -------------------------------------------------------------------
empty_cols = df_close_full.columns[df_close_full.isna().all()].tolist()
df_close_full = df_close_full.drop(columns=empty_cols)

# Keep only tickers with enough history to support a 200-day moving average
min_obs = 260
short_history = df_close_full.columns[df_close_full.notna().sum() < min_obs].tolist()
df_close_full = df_close_full.drop(columns=short_history)

# Forward-fill isolated holes (single missing prints), never back-fill (that would look ahead)
df_close_full = df_close_full.ffill(limit=3)

print(f"Tickers requested            : {len(yahoo_symbols)}")
print(f"Tickers with no data at all  : {len(empty_cols)} {empty_cols}")
print(f"Tickers with < {min_obs} prints  : {len(short_history)} {short_history}")
print(f"Tickers kept for the analysis: {df_close_full.shape[1]} "
      f"({df_close_full.shape[1] / len(yahoo_symbols):.1%} coverage)")

In [ ]:
# The deliverable: closing prices over the required 1-year analysis window
df_close = df_close_full.loc[start_date:end_date].copy()
df_close.index.name = 'DATE'

print(f"df_close: {df_close.shape[0]} trading days x {df_close.shape[1]} stocks "
      f"({df_close.index.min().date()} -> {df_close.index.max().date()})")
display(df_close)

In [ ]:
# Sanity check: no all-NaN rows, and the % of missing values is negligible
print(f"Missing values in df_close: {df_close.isna().to_numpy().mean():.3%}")
display(df_close.iloc[:, :8].describe().T.head(8))

## Identify Golden and Death Crosses

### Get Moving Averages 50 days and 200 days

The moving averages are computed on the **full** (warm-up + analysis) price history so that MA200 is
already "warm" on 2024-05-01, and are then sliced back to the analysis window. `min_periods` is set
equal to the window so that no average is produced from an incomplete look-back — a partially filled
MA200 would create phantom crosses at the start of the sample.

In [ ]:
# Compute moving averages on the full history (warm-up included), then slice to the analysis window
df_ma50_full = df_close_full.rolling(window=50, min_periods=50).mean()
df_ma200_full = df_close_full.rolling(window=200, min_periods=200).mean()

df_ma50 = df_ma50_full.loc[start_date:end_date].copy()
df_ma200 = df_ma200_full.loc[start_date:end_date].copy()
df_ma50.index.name = df_ma200.index.name = 'DATE'

print(f"df_ma50 : {df_ma50.shape} | first valid date: {df_ma50.dropna(how='all').index.min().date()}")
print(f"df_ma200: {df_ma200.shape} | first valid date: {df_ma200.dropna(how='all').index.min().date()}")

In [ ]:
display(df_ma50)

In [ ]:
display(df_ma200)

In [ ]:
# Coverage check: both indicators must be defined for (almost) every stock on every day of the window
coverage = pd.DataFrame({
    'MA50_defined_%': df_ma50.notna().mean(axis=1),
    'MA200_defined_%': df_ma200.notna().mean(axis=1),
})
display(coverage.iloc[[0, 1, -2, -1]].style.format('{:.1%}'))

### Detecting Golden and Death Crosses in the last 14 days

**Definition used**

* **Golden Cross** — on day *t* the MA50 closes **above** the MA200 while on day *t-1* it was at or
  below it: `MA50[t-1] <= MA200[t-1]` and `MA50[t] > MA200[t]`.
* **Death Cross** — the mirror image: `MA50[t-1] >= MA200[t-1]` and `MA50[t] < MA200[t]`.

Both averages must be defined on *t* and *t-1*, otherwise the "cross" is just the indicator coming
into existence. The function below is vectorised over the whole (dates × tickers) matrix and returns
one tidy row per event, which is the format we need for filtering, plotting and back-testing.

In [ ]:
def detect_crosses(df_ma_short, df_ma_long, df_price=None):
    """Detect every MA50/MA200 crossover for every stock.

    A Golden Cross happens when the short MA moves from at-or-below to strictly above the long MA;
    a Death Cross is the symmetric event. Both averages must exist on the event day and the day
    before, so that the first valid MA200 print is never mistaken for a cross.

    Returns a tidy DataFrame with one row per (ticker, cross date) event.
    """
    short = df_ma_short.to_numpy(dtype=float)
    long = df_ma_long.to_numpy(dtype=float)

    diff = short - long
    prev = np.vstack([np.full((1, diff.shape[1]), np.nan), diff[:-1]])   # diff shifted by one day
    valid = ~np.isnan(diff) & ~np.isnan(prev)

    golden = valid & (prev <= 0) & (diff > 0)
    death = valid & (prev >= 0) & (diff < 0)

    dates = df_ma_short.index
    tickers = df_ma_short.columns
    price = df_price.reindex(index=dates, columns=tickers).to_numpy(dtype=float) \
        if df_price is not None else None

    rows = []
    for mask, label in ((golden, 'GOLDEN'), (death, 'DEATH')):
        for i, j in np.argwhere(mask):
            rows.append({
                'TICKER': tickers[j],
                'CROSS_TYPE': label,
                'CROSS_DATE': dates[i],
                'CLOSE_AT_CROSS': price[i, j] if price is not None else np.nan,
                'MA50_AT_CROSS': short[i, j],
                'MA200_AT_CROSS': long[i, j],
                'MA_SPREAD_PCT': (short[i, j] - long[i, j]) / long[i, j] * 100,
            })

    cols = ['TICKER', 'CROSS_TYPE', 'CROSS_DATE', 'CLOSE_AT_CROSS',
            'MA50_AT_CROSS', 'MA200_AT_CROSS', 'MA_SPREAD_PCT']
    out = pd.DataFrame(rows, columns=cols)
    if out.empty:
        return out
    return out.sort_values(['CROSS_DATE', 'TICKER']).reset_index(drop=True)


df_crosses = detect_crosses(df_ma50, df_ma200, df_close)
print(f"Crosses detected over {start_date} -> {end_date}: {len(df_crosses)}")
print(df_crosses['CROSS_TYPE'].value_counts().to_string())
display(df_crosses.head(10))

In [ ]:
# How many crossovers per month? (a first sanity check on the market regime)
monthly = (df_crosses.assign(MONTH=df_crosses['CROSS_DATE'].dt.to_period('M'))
                     .pivot_table(index='MONTH', columns='CROSS_TYPE',
                                  values='TICKER', aggfunc='count')
                     .fillna(0).astype(int))
display(monthly)

In [ ]:
# --- Filter: only the crosses that happened in the LAST 14 DAYS of the sample ------------------
last_date = df_close.index.max()
window_start = last_date - pd.Timedelta(days=14)          # 14 calendar days (~10 trading sessions)

df_crosses_14d = df_crosses[df_crosses['CROSS_DATE'] > window_start].copy()

print(f"Last date in the sample : {last_date.date()}")
print(f"14-day window           : {window_start.date()} -> {last_date.date()} "
      f"({df_close.loc[window_start:last_date].shape[0]} trading sessions)")
print(f"Crosses inside the window: {len(df_crosses_14d)}")
display(df_crosses_14d)

#### Enrich the signals with volatility and company metadata

Two volatility measures are attached to every signal, because they answer different questions:

* `VOL_ANNUAL_1Y` — annualised standard deviation of daily returns over the full year
  (`std(daily returns) × √252`): *how risky is this stock in general?*
* `VOL_30D_AROUND_CROSS` — the same measure computed on a ±15-session window centred on the cross:
  *how agitated was the stock exactly when the signal fired?*

In [ ]:
# Compute the volatility of every stock in the S&P 500
df_returns = df_close.pct_change()
vol_annual = df_returns.std() * np.sqrt(252)              # annualised realised volatility
vol_annual.name = 'VOL_ANNUAL_1Y'

display(vol_annual.sort_values(ascending=False).head(10).to_frame().style.format('{:.2%}'))
print(f"S&P 500 median annualised volatility: {vol_annual.median():.2%}")

In [ ]:
def local_volatility(ticker, cross_date, sessions=15):
    """Annualised volatility on a +/- `sessions` window centred on the cross date."""
    idx = df_returns.index
    pos = idx.get_indexer([cross_date], method='nearest')[0]
    window = df_returns[ticker].iloc[max(0, pos - sessions): pos + sessions + 1]
    return window.std() * np.sqrt(252)


def enrich(df_signals):
    if df_signals.empty:
        return df_signals
    out = df_signals.copy()
    out['VOL_ANNUAL_1Y'] = out['TICKER'].map(vol_annual)
    out['VOL_30D_AROUND_CROSS'] = [local_volatility(t, d)
                                   for t, d in zip(out['TICKER'], out['CROSS_DATE'])]
    out['COMPANY'] = out['TICKER'].map(meta['COMPANY'])
    out['SECTOR'] = out['TICKER'].map(meta['SECTOR'])
    return out


df_crosses_14d = enrich(df_crosses_14d)

df_golden_cross_14d = (df_crosses_14d[df_crosses_14d['CROSS_TYPE'] == 'GOLDEN']
                       .sort_values('TICKER').reset_index(drop=True))
df_death_cross_14d = (df_crosses_14d[df_crosses_14d['CROSS_TYPE'] == 'DEATH']
                      .sort_values('TICKER').reset_index(drop=True))

print(f"Golden crosses in the last 14 days: {len(df_golden_cross_14d)}")
print(f"Death crosses  in the last 14 days: {len(df_death_cross_14d)}")

In [ ]:
# use the display function to show as many intermediary results
# for example display(df_golden_cross_14d)
display(df_golden_cross_14d.style.format({
    'CLOSE_AT_CROSS': '{:.2f}', 'MA50_AT_CROSS': '{:.2f}', 'MA200_AT_CROSS': '{:.2f}',
    'MA_SPREAD_PCT': '{:+.2f}%', 'VOL_ANNUAL_1Y': '{:.2%}', 'VOL_30D_AROUND_CROSS': '{:.2%}'}))

In [ ]:
# use the display function to show as many intermediary results
# for example display(df_death_cross_14d)
display(df_death_cross_14d.style.format({
    'CLOSE_AT_CROSS': '{:.2f}', 'MA50_AT_CROSS': '{:.2f}', 'MA200_AT_CROSS': '{:.2f}',
    'MA_SPREAD_PCT': '{:+.2f}%', 'VOL_ANNUAL_1Y': '{:.2%}', 'VOL_30D_AROUND_CROSS': '{:.2%}'}))

In [ ]:
# Sector breakdown of the recent signals — is the market rotating, or is it a broad move?
if not df_crosses_14d.empty:
    display(pd.crosstab(df_crosses_14d['SECTOR'], df_crosses_14d['CROSS_TYPE']))

#### Golden crosses
List the first top companies in alphabetical order (by there symbol or ticker) that had a golden cross in the last 14 days:

In [ ]:
top_golden = df_golden_cross_14d['TICKER'].sort_values().unique().tolist()[:10]
top_death = df_death_cross_14d['TICKER'].sort_values().unique().tolist()[:10]

display(Markdown(
    "**Top 10 GOLDEN crosses (alphabetical order):** "
    + (", ".join(f"`{t}`" for t in top_golden) if top_golden else "_none in the window_")
))
display(df_golden_cross_14d.loc[df_golden_cross_14d['TICKER'].isin(top_golden),
                                ['TICKER', 'COMPANY', 'SECTOR', 'CROSS_DATE',
                                 'CLOSE_AT_CROSS', 'VOL_ANNUAL_1Y']]
        .reset_index(drop=True))

**Answer.** The ten S&P 500 companies (alphabetical order by ticker) that produced a **Golden Cross**
in the 14 days ending on the last date of the sample are the ones printed by the cell above — the
list is generated programmatically from `df_golden_cross_14d` so that it can never drift away from
the data.

> 📋 **After running the notebook, paste the printed tickers here** so the answer is also readable
> without executing the cells: `1. … 2. … 3. …`

Note that the exact membership of this list depends on the date on which the notebook is run and on
how many constituents Yahoo Finance returns that day: a cross is a *point-in-time* event, so re-running
the notebook a week later will legitimately produce a different set of names.

#### Death crosses
List the first 10 companies in alphabetical order (by there symbol or ticker) that had a death cross in the last 14 days:

In [ ]:
display(Markdown(
    "**Top 10 DEATH crosses (alphabetical order):** "
    + (", ".join(f"`{t}`" for t in top_death) if top_death else "_none in the window_")
))
display(df_death_cross_14d.loc[df_death_cross_14d['TICKER'].isin(top_death),
                               ['TICKER', 'COMPANY', 'SECTOR', 'CROSS_DATE',
                                'CLOSE_AT_CROSS', 'VOL_ANNUAL_1Y']]
        .reset_index(drop=True))

**Answer.** The ten S&P 500 companies (alphabetical order by ticker) that produced a **Death Cross**
in the last 14 days of the sample are printed by the cell above, taken directly from
`df_death_cross_14d`.

> 📋 **After running the notebook, paste the printed tickers here:** `1. … 2. … 3. …`

Death crosses tend to cluster in the same sectors (they are driven by a common factor — rates,
oil, a sector-wide de-rating), which is exactly what the sector cross-tab above is designed to reveal.

### Visualization of the results
(in alphabetical order)

#### Compute the volatility of every stock and print it in the title of each plot 

In [ ]:
def plot_cross(ticker, cross_rows, ax=None):
    """Price + MA50 + MA200 for one stock, with the 14-day window shaded and the crosses marked."""
    if ax is None:
        _, ax = plt.subplots(figsize=(12, 5))

    close, ma50, ma200 = df_close[ticker], df_ma50[ticker], df_ma200[ticker]

    # Leave head/foot room so the annotations never collide with the title or the axis
    lo, hi = np.nanmin(close.to_numpy()), np.nanmax(close.to_numpy())
    pad = (hi - lo) * 0.12
    ax.set_ylim(lo - pad, hi + pad)

    ax.plot(close.index, close, color='#4c72b0', lw=1.3, label='Close price')
    ax.plot(ma50.index, ma50, color='#dd8452', lw=1.6, label='MA50')
    ax.plot(ma200.index, ma200, color='#55a868', lw=1.6, label='MA200')

    # Shade the last 14 days: the window in which we are looking for signals
    ax.axvspan(window_start, last_date, color='gold', alpha=0.20,
               label='Last 14 days (signal window)')

    for _, row in cross_rows.iterrows():
        is_golden = row['CROSS_TYPE'] == 'GOLDEN'
        colour = '#2ca02c' if is_golden else '#d62728'
        ax.axvline(row['CROSS_DATE'], color=colour, ls='--', lw=1.2, alpha=0.9)
        ax.scatter([row['CROSS_DATE']], [row['CLOSE_AT_CROSS']],
                   marker='^' if is_golden else 'v', s=160, zorder=5,
                   color=colour, edgecolor='black', linewidth=0.6,
                   label=f"{row['CROSS_TYPE'].title()} Cross "
                         f"({row['CROSS_DATE'].date()})")
        # Push the label downwards when the cross sits high in the chart, and vice versa
        rel = (row['CLOSE_AT_CROSS'] - lo) / (hi - lo) if hi > lo else 0.5
        dy = -48 if rel > 0.55 else 32
        ax.annotate(f"{row['CROSS_TYPE'].title()} Cross\n{row['CROSS_DATE'].date()}",
                    xy=(row['CROSS_DATE'], row['CLOSE_AT_CROSS']),
                    xytext=(-100, dy), textcoords='offset points',
                    fontsize=8, color=colour, fontweight='bold',
                    arrowprops=dict(arrowstyle='->', color=colour, lw=0.8))

    row0 = cross_rows.iloc[0]
    ax.set_title(
        f"{ticker} — {row0['COMPANY']} ({row0['SECTOR']})\n"
        f"{row0['CROSS_TYPE'].title()} Cross on {row0['CROSS_DATE'].date()}  |  "
        f"Annualised volatility: {row0['VOL_ANNUAL_1Y']:.1%} (1Y)  ·  "
        f"{row0['VOL_30D_AROUND_CROSS']:.1%} (30 sessions around the cross)",
        fontsize=11, fontweight='bold')
    ax.set_xlabel('Date')
    ax.set_ylabel('Adjusted close price (USD)')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
    ax.legend(loc='best', fontsize=8, framealpha=0.9)
    ax.margins(x=0.01)
    return ax


def plot_top10(df_signals, kind):
    """Plot the first 10 tickers (alphabetical order) carrying a given signal."""
    tickers = df_signals['TICKER'].sort_values().unique().tolist()[:10]
    if not tickers:
        print(f"No {kind} cross detected in the last 14 days — nothing to plot.")
        return
    print(f"Plotting {len(tickers)} {kind} cross stocks (alphabetical order): {', '.join(tickers)}\n")
    for ticker in tickers:
        rows = df_signals[df_signals['TICKER'] == ticker]
        fig, ax = plt.subplots(figsize=(12, 5))
        plot_cross(ticker, rows, ax=ax)
        plt.tight_layout()
        plt.show()

#### Plot top 10 stocks that had Golden Crosses in the last 14 days

- You should have 10 plots (use a for loop) for every stock in the top 10 (in alphabetical order)
- For each plot, put the volatility of the stock in the title of the plot

In [ ]:
plot_top10(df_golden_cross_14d, 'GOLDEN')

### Plot top 10 stocks that had Death Crosses in the last 14 days

You should have 10 plots (use a for loop) for every stock in the top 10 (in alphabetical order)
For each plot, put the volatility of the stock in the title of the plot

In [ ]:
plot_top10(df_death_cross_14d, 'DEATH')

#### Volatility in context

A single number in a title is hard to judge, so the chart below places the volatility of the signalled
stocks against the distribution of the whole index. If the crossing names sit systematically in the
right tail, the "signal" may be little more than noisy stocks whipsawing around their own averages.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.hist(vol_annual.dropna(), bins=40, color='#c7cdd6', edgecolor='white',
        label='S&P 500 constituents')
ax.axvline(vol_annual.median(), color='black', ls='--', lw=1.5,
           label=f"Index median ({vol_annual.median():.1%})")

for df_sig, colour, label in ((df_golden_cross_14d, '#2ca02c', 'Golden cross (14d)'),
                              (df_death_cross_14d, '#d62728', 'Death cross (14d)')):
    if not df_sig.empty:
        ax.plot(df_sig['VOL_ANNUAL_1Y'], np.full(len(df_sig), 2), '|', ms=22, mew=2,
                color=colour, label=f"{label} — median {df_sig['VOL_ANNUAL_1Y'].median():.1%}")

ax.set_title('Annualised volatility: stocks with a recent cross vs. the rest of the index',
             fontweight='bold')
ax.set_xlabel('Annualised volatility (std of daily returns x sqrt(252))')
ax.set_ylabel('Number of stocks')
ax.legend()
plt.tight_layout()
plt.show()

---
## Backtesting the signal (event study)

Answering *"does a Golden Cross actually make money?"* requires more than looking at the last 14 days,
so we run a simple **event study** on every cross detected during the year: for each signal we measure
the forward return over 5, 10, 21 and 63 trading sessions and compare it with the **unconditional**
return of the same stocks over the same horizons (the "do nothing special" benchmark).

Caveats, stated up front: one year of data on a single (bullish) regime is a very small sample, the
forward window is truncated for the most recent signals, and the study ignores transaction costs and
survivorship bias — today's S&P 500 membership list excludes companies that were dropped from the
index during the period.

In [ ]:
HORIZONS = [5, 10, 21, 63]


def forward_returns(df_signals, horizons=HORIZONS):
    """Return, for each cross, the forward return over each horizon (in trading sessions)."""
    out = df_signals.copy()
    idx = df_close.index
    for h in horizons:
        rets = []
        for ticker, date in zip(out['TICKER'], out['CROSS_DATE']):
            pos = idx.get_indexer([date])[0]
            if pos == -1 or pos + h >= len(idx):
                rets.append(np.nan)                       # not enough future data yet
                continue
            p0, p1 = df_close[ticker].iloc[pos], df_close[ticker].iloc[pos + h]
            rets.append(p1 / p0 - 1 if pd.notna(p0) and pd.notna(p1) else np.nan)
        out[f'FWD_{h}D'] = rets
    return out


df_events = forward_returns(enrich(df_crosses))
print(f"Events studied: {len(df_events)} "
      f"({(df_events['CROSS_TYPE'] == 'GOLDEN').sum()} golden / "
      f"{(df_events['CROSS_TYPE'] == 'DEATH').sum()} death)")
display(df_events.head())

In [ ]:
# Benchmark: unconditional forward returns of the same universe over the same horizons
baseline = {f'FWD_{h}D': (df_close.shift(-h) / df_close - 1).stack().mean() for h in HORIZONS}

summary = []
for cross_type in ['GOLDEN', 'DEATH']:
    sub = df_events[df_events['CROSS_TYPE'] == cross_type]
    for h in HORIZONS:
        col = f'FWD_{h}D'
        series = sub[col].dropna()
        summary.append({
            'CROSS_TYPE': cross_type,
            'HORIZON_DAYS': h,
            'N_EVENTS': len(series),
            'MEAN_RETURN': series.mean(),
            'MEDIAN_RETURN': series.median(),
            'HIT_RATE': (series > 0).mean(),
            'BASELINE_MEAN': baseline[col],
            'EXCESS_VS_BASELINE': series.mean() - baseline[col],
        })

df_backtest = pd.DataFrame(summary)
display(df_backtest.style.format({
    'MEAN_RETURN': '{:+.2%}', 'MEDIAN_RETURN': '{:+.2%}', 'HIT_RATE': '{:.1%}',
    'BASELINE_MEAN': '{:+.2%}', 'EXCESS_VS_BASELINE': '{:+.2%}'}))

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
width = 0.35
x = np.arange(len(HORIZONS))

for offset, (cross_type, colour) in zip((-width / 2, width / 2),
                                        (('GOLDEN', '#2ca02c'), ('DEATH', '#d62728'))):
    vals = [df_backtest.query("CROSS_TYPE == @cross_type and HORIZON_DAYS == @h")['MEAN_RETURN'].iloc[0]
            for h in HORIZONS]
    ax.bar(x + offset, vals, width, color=colour, label=f'{cross_type.title()} cross', alpha=0.85)

ax.plot(x, [baseline[f'FWD_{h}D'] for h in HORIZONS], 'k--o', lw=1.5,
        label='Unconditional baseline (all stocks, all days)')
ax.axhline(0, color='black', lw=0.8)
ax.set_xticks(x, [f'{h} sessions' for h in HORIZONS])
ax.set_ylabel('Mean forward return')
ax.set_title('Mean forward return after a cross vs. the unconditional baseline',
             fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Are the differences even distinguishable from noise? A quick two-sample t-test per horizon.
from scipy import stats

for h in HORIZONS:
    g = df_events.query("CROSS_TYPE == 'GOLDEN'")[f'FWD_{h}D'].dropna()
    d = df_events.query("CROSS_TYPE == 'DEATH'")[f'FWD_{h}D'].dropna()
    if len(g) > 2 and len(d) > 2:
        t, p = stats.ttest_ind(g, d, equal_var=False)
        verdict = 'significant at 5%' if p < 0.05 else 'NOT significant at 5%'
        print(f"{h:>3} sessions | golden {g.mean():+.2%} (n={len(g)}) vs death {d.mean():+.2%} "
              f"(n={len(d)}) | t={t:+.2f}, p={p:.3f} -> {verdict}")

## Question section

### Understanding concepts

#### What is a Golden Cross and what does it typically signal to investors?

A **Golden Cross** occurs when a short-term moving average crosses **above** a long-term moving
average — in the standard formulation used here, when the 50-day simple moving average of the closing
price rises above the 200-day simple moving average. Mechanically it means that the average price
paid over the last ~2.5 months has moved above the average price paid over the last ~10 months: recent
buyers are, on aggregate, paying more than longer-term holders, and the trend of the medium term has
turned up relative to the long term.

To investors it is read as a **confirmation of a bullish regime change**, not as a prediction. Three
things are usually inferred from it:

1. **Trend confirmation.** The down-leg has ended and momentum has flipped; trend-following systems
   move from neutral/short to long.
2. **A moving stop level.** The MA200 becomes the reference line: as long as price holds above it, the
   long thesis is intact, which gives a mechanical, non-emotional exit rule.
3. **A crowding/reflexivity effect.** Because the signal is universally known and published by every
   financial data provider, a Golden Cross on a large name attracts flow from CTAs and momentum funds,
   and generates media coverage that can be partially self-fulfilling in the short term.

Its defining characteristic is that it is a **lagging** indicator: it is built from 50 and 200 days of
history, so by construction it confirms a move that has already happened — typically the price has
already risen 10–25% off the low by the time the cross prints. It buys *confirmation* at the cost of
*timeliness*.

#### What is a Death Cross and how might market participants react to it?

A **Death Cross** is the symmetric event: the MA50 falls **below** the MA200, meaning the recent
average price has dropped under the long-term average and the medium-term trend has turned down. It is
interpreted as confirmation that a correction has become a **downtrend** rather than a dip.

Reactions differ sharply by type of participant, which is important to understand because it explains
why the signal moves prices at all:

* **Systematic trend-followers (CTAs, managed futures)** flip to flat or short mechanically, which is
  the main channel through which the signal becomes partially self-fulfilling.
* **Risk managers and long-only funds** rarely sell purely on a cross, but use it to cut position
  sizes, tighten stops, or hedge with options — the cross acts as a trigger for a *review*, not for a
  liquidation.
* **Retail investors** are the most reactive: "death cross" is a headline-friendly phrase, and its
  appearance in the financial press often coincides with a spike in selling volume.
* **Contrarian and value investors** frequently do the opposite: because the indicator is so lagging,
  a Death Cross often prints *near* a local bottom, after the bulk of the decline is already priced in
  (the March 2020 death crosses printed within days of the low).

The practical caveat is the **whipsaw**: in a choppy, range-bound market the two averages can cross
back and forth repeatedly, generating a Death Cross that is reversed by a Golden Cross a few weeks
later, and each round trip costs spread, commission and taxes.

#### Why might moving averages (MA50, MA200) be used as indicators in technical analysis?

Moving averages are used because they solve a very specific problem: **daily prices are dominated by
noise**, and the human eye (and most statistical rules) cannot separate a trend from that noise
without smoothing. A moving average is the simplest possible low-pass filter — it removes
high-frequency variation and leaves the slow-moving component that trend-following seeks to exploit.

Concretely they are attractive because:

* **They are objective and reproducible.** Unlike "head and shoulders" or trendline patterns, an MA
  crossover is fully mechanical: two analysts computing MA50/MA200 on the same data get identical
  signals. This is what makes them backtestable.
* **They summarise cost basis.** The MA200 is a decent proxy for the average price paid by holders over
  the past year; price above it means the median holder is in profit — which has real behavioural
  consequences for selling pressure.
* **The horizons are meaningful.** ~50 trading days ≈ one quarter (one earnings cycle); ~200 trading
  days ≈ one trading year. The two windows therefore compare "the current quarter's regime" with "the
  annual regime", which maps onto how institutional allocations are actually reviewed.
* **They are self-reinforcing (a Schelling point).** Because so many participants watch the same two
  lines, they become de-facto support/resistance levels. Their popularity is itself a source of their
  (limited) predictive power.
* **They are trivially cheap to compute** for thousands of assets — as this notebook shows, the whole
  index is two `rolling().mean()` calls.

The counterpart of the smoothing is **lag**: an N-day simple moving average lags the underlying series
by roughly N/2 days. MA200 therefore reacts to a turn with roughly a 100-day delay, which is exactly
why crossovers confirm trends rather than anticipate them, and why they perform well in persistent
trends and badly in range-bound markets.

#### Why are the last 14 days used to check for crosses? What are the implications of this choice?

The 14-day filter exists to make the output **actionable**. A crossover detected 8 months ago is
history: the move it announced has already played out, and it tells a trader nothing about what to do
today. Restricting attention to ~14 calendar days (≈10 trading sessions) keeps only signals that are
still "fresh" — recent enough that the entry price is close to today's price, but old enough (a few
sessions) to have survived an immediate reversal. It also keeps the output list short enough to be
reviewed by a human analyst, which matters when screening 500 names.

**Implications and trade-offs:**

* **Calendar days ≠ trading days.** 14 calendar days contain only ~10 sessions, and fewer when a
  holiday falls inside them. The size of the sample therefore varies with where the window lands in the
  calendar; using `BDay(10)` would make the window stable, at the cost of a less intuitive definition.
* **Small, unstable samples.** In a quiet market the window can be empty; after a sharp regime shift it
  can contain dozens of names at once. Any statistic computed on the filtered set (e.g. average
  volatility of the "signal group") is therefore very noisy — this is a screening tool, not a sample
  for inference. This is precisely why the back-test above uses *all* crosses of the year, not the
  14-day subset.
* **No confirmation period.** 14 days is short enough that some of the crosses in the list are
  *unconfirmed*: the averages may cross back within days. A common professional refinement is to
  require the spread between the averages to stay positive for k consecutive sessions, or to exceed a
  minimum percentage (`MA_SPREAD_PCT` in our table is designed for exactly this filter) — at the price
  of entering later.
* **Look-ahead risk if implemented carelessly.** The window is defined relative to the last date of the
  sample; in a live system it must be re-anchored to "today", and the cross must be computed on
  *closed* bars only, otherwise an intraday print can create a signal that disappears at the close.
* **Endpoint sensitivity.** Because the window is anchored to the end of the data, the answer to
  "which stocks crossed?" is not stable through time — re-running this notebook a week later
  legitimately produces a different list. That is a property of the screen, not a bug, but it must be
  disclosed when the results are presented.

#### How does volatility (e.g., measured using percentage change standard deviation) help contextualize the price movement around crosses?

Volatility is the **denominator** that turns a raw signal into a risk-adjusted one. A crossover in a
stock whose annualised volatility is 15% and the same crossover in a stock at 70% are not comparable
events, for three reasons:

1. **It calibrates the reliability of the signal.** Moving averages are estimates of a trend, and their
   sampling error scales with volatility. In a high-volatility name the MA50 and MA200 sit close
   together in units of daily standard deviation, so they cross frequently and largely at random — the
   classic whipsaw. A cross in a 15%-vol utility carries much more information per event than a cross
   in a 70%-vol biotech. Normalising the gap between the averages by volatility
   (`MA_SPREAD_PCT / VOL`) is the natural way to rank signals by conviction, and it is why the two
   volatility columns are attached to every signal in `df_crosses_14d`.
2. **It sizes the position and the stop.** Practitioners do not buy "100 shares"; they buy an amount
   such that a 1-σ adverse move costs a fixed fraction of the book (volatility targeting). The
   annualised volatility in the plot titles is what converts "buy the Golden Cross" into an actual
   trade size, and it sets the distance at which a stop can be placed without being hit by ordinary
   noise.
3. **It describes the state of the market around the event.** Comparing `VOL_30D_AROUND_CROSS` with
   `VOL_ANNUAL_1Y` says whether the cross happened in a calm, orderly trend (local vol ≤ 1-year vol —
   typically a healthier signal) or inside a stress episode (local vol well above the 1-year level),
   where the crossover is more likely to be an artefact of a violent move that will mean-revert. Rising
   volatility around a Golden Cross is a warning that the "trend" is really a high-variance bounce.

The histogram plotted above adds the population view: it shows whether the signalled names are ordinary
members of the index or systematically the most volatile ones — if the latter, part of any measured
"edge" is simply compensation for higher risk rather than genuine predictive power, and any comparison
of returns must be risk-adjusted (Sharpe, or returns per unit of volatility) rather than raw.

### Backtesting and evaluation

#### How would you measure whether Golden Crosses actually lead to profitable trades?

By running a disciplined **event study** first and a **portfolio back-test** second — the two answer
different questions, and the notebook implements the first one above.

**1. Event study (does the signal carry information?)**
For every Golden Cross in a long history, measure the forward return over several horizons (5, 10, 21,
63 sessions — the `FWD_*` columns computed above) and compare it with a benchmark. The benchmark is the
crux: the raw average return after a Golden Cross is positive mostly because *stocks go up on average*.
Valid controls are (a) the unconditional mean return of the same universe over the same horizons —
implemented above as `BASELINE_MEAN`; (b) the contemporaneous return of the S&P 500 index, to strip out
market beta; (c) a matched sample of non-signalling stocks with similar sector and volatility.
Report mean **and** median (returns are skewed), the **hit rate**, and a test of statistical
significance — the Welch t-test in the cell above is the minimum, though returns are non-normal and
overlapping windows make observations dependent, so a bootstrap or block-bootstrap over event dates is
the honest version.

**2. Portfolio back-test (would the rule have made money?)**
Turn the signal into a full trading rule and simulate it end to end: enter at the *next* session's open
after the cross (never at the closing price that generated it — that is look-ahead bias), define an
exit (a Death Cross, a fixed horizon, a trailing stop), size positions by volatility, cap the number of
concurrent positions, and subtract realistic costs (commission + half the bid-ask spread + slippage,
typically 5–15 bps per side for S&P 500 names). Then evaluate: CAGR, volatility, **Sharpe and Sortino**,
maximum drawdown, turnover, hit rate, average win/average loss, and — most importantly — the return
**relative to buy-and-hold on SPY**, since a long-only trend rule in a bull market will look brilliant
in absolute terms while underperforming a passive index.

**3. Validation discipline (is the result real?)**
Split the history into in-sample and out-of-sample periods and check the rule survives both; test
across regimes (2008, 2018, 2020, 2022) rather than a single bull year; correct for **survivorship
bias** by using the *historical* index membership rather than today's constituents; account for
overlapping events; and count every parameter you tried (50/200 vs 20/100 vs 10/50) — testing many
combinations and reporting the best one is data-snooping, and a Deflated Sharpe Ratio or White's
Reality Check is the proper adjustment.

Applied to this notebook, the honest conclusion is bounded by the data: one year, one regime, ~a few
hundred events, forward windows truncated for the most recent signals. That is enough to *illustrate*
the methodology and to reject the strongest claims, not enough to validate a strategy.

#### What are the risks of using only technical indicators like moving averages without incorporating fundamentals?

* **No notion of value — you can buy an expensive falling knife or sell a cheap compounding asset.**
  A moving average knows nothing about earnings, leverage or cash flow. A Golden Cross on a company
  whose margins are collapsing is a momentum artefact; a Death Cross on a solid business trading at 10×
  earnings is often a gift to a value buyer. Price alone cannot distinguish the two.
* **Blindness to the catalyst, and therefore to the risk.** The rule cannot tell whether a downtrend is
  caused by a temporary logistics problem or by a solvency issue, an accounting fraud or a patent
  cliff. In the tail cases (Enron, Wirecard, SVB, Credit Suisse) fundamentals and news gave a warning
  that price smoothing structurally lags by ~100 days.
* **Event risk that no MA can anticipate.** Earnings, guidance cuts, FDA rulings, M&A, index inclusion
  and macro prints move prices by multiples of daily volatility in a single gap. Gaps jump *over* stop
  levels, so a strategy whose risk control is "the MA200 is my stop" is not actually protected.
* **Whipsaw and cost drag in range-bound markets.** Trend rules are profitable in ~30% of the time
  (persistent trends) and bleed in the rest. Without a regime or trend-strength filter (ADX, realised
  vol, distance between averages), the accumulated round-trip costs and taxes can exceed the gross edge.
* **Structural false positives.** Splits, special dividends, corporate actions and thin liquidity all
  produce mechanical crossovers with no economic content — using adjusted closes (as here,
  `auto_adjust=True`) removes most, but not all, of them.
* **Crowding.** Precisely because it is famous and free, the MA50/MA200 crossover is arbitraged by
  faster participants. Any edge that exists is thin, decays as it becomes better known, and is the
  first to be front-run.
* **A false sense of rigour.** Numerical output looks scientific. Backtested on a single bull year,
  almost any long-only rule will show attractive statistics — the danger is confusing *quantified* with
  *validated*.

The reasonable conclusion is not that technical analysis is useless, but that MA crossovers are a
**timing and risk-management layer**, most defensible when combined with a fundamental or
sentiment-based view of *what* to trade, while the technicals decide *when* and *how much*.

#### How would you improve this strategy to reduce false signals (e.g., a Golden Cross that doesn’t lead to a price increase)?

**A. Make the signal itself harder to trigger**

1. **Minimum separation (noise band).** Require `MA50 > MA200 × (1 + δ)` with δ ≈ 0.5–1%, instead of a
   bare crossing. The `MA_SPREAD_PCT` column already computed makes this a one-line filter and removes
   the marginal crossings that reverse immediately.
2. **Volatility-normalised threshold.** Better than a fixed δ: require the spread to exceed
   *k × daily volatility*, so the bar adapts to each stock (a 1% spread means something very different
   for a utility and for a biotech).
3. **Confirmation delay.** Require the condition to hold for 3–5 consecutive sessions, or that price
   close above the MA200 as well. This mechanically eliminates one-day whipsaws at the cost of a later
   entry.
4. **Exponential or double-smoothed averages.** EMAs react faster than SMAs for the same window;
   pairing them with a slope condition (`MA200` must itself be flat-to-rising for a Golden Cross to
   count) rejects crosses that occur inside an established downtrend.

**B. Add confirmation from independent information**

5. **Volume confirmation.** A cross accompanied by above-average volume (or a rising OBV) is far more
   likely to reflect real institutional repositioning than one on thin volume.
6. **Market and sector regime filter.** Only take long signals when the index itself is above its own
   MA200, and prefer stocks whose sector is outperforming — a single-name Golden Cross in a bear
   market fails far more often.
7. **Cross-indicator agreement.** Require RSI or MACD not to be in extreme/divergent territory, and use
   relative strength versus the index rather than absolute price for the crossover.
8. **Fundamental and news screens.** Exclude names with deteriorating earnings revisions, or with
   pending binary events; and — the bridge to Section A — use news embeddings/sentiment to check that
   the move is not driven by a one-off headline. A Golden Cross with improving fundamentals and
   positive news flow is a materially different trade from a technically identical one on a company
   under investigation.

**C. Manage the trade, not just the entry**

9. **Volatility-based sizing and stops** (ATR multiples), so no single false positive can dominate the
   P&L; and diversification limits per sector.
10. **Predefined exit rules** — a Death Cross, a time stop (if the trade has not worked in N sessions,
    the signal has failed), or a trailing stop — since most of the damage from false positives comes
    from holding them, not from entering them.

**D. Validate honestly.** Every filter added above is a parameter, and every parameter is an
opportunity to overfit. Each one should be justified *ex ante*, tested out-of-sample and across
regimes, and evaluated on the basis of whether it improves the Sharpe ratio net of costs — not merely
whether it removes signals that failed in hindsight.

### AI Integration

#### Could sentiment from news (future project part) help validate or invalidate these technical signals?

Yes, and mainly by supplying the one thing a moving average structurally cannot: the **cause** of the
move and its **timeliness**. Price smoothing lags by design; news arrives first. Combining them is
therefore complementary rather than redundant.

**How it validates or invalidates a signal:**

* **Corroboration.** A Golden Cross accompanied by a cluster of positive, *fundamental* news — earnings
  beats, guidance raises, contract wins, analyst upgrades — suggests the trend has an economic driver
  and is more likely to persist. That is a "confirmed" signal.
* **Contradiction.** A Golden Cross while the news flow is negative or dominated by speculation
  (short squeeze, meme-driven flows, takeover rumours that later evaporate) is a red flag: the trend
  lacks fundamental support and is more likely to reverse. Symmetrically, a Death Cross with intact
  fundamentals and merely macro-driven weakness may be a buying opportunity for a longer-horizon
  investor rather than an exit.
* **Event classification.** The clustering built in Section A is directly useful here: knowing that a
  stock's recent news falls in the "regulatory/legal" cluster rather than the "product/earnings"
  cluster changes the interpretation of an identical technical picture. Sentiment gives direction;
  the cluster gives the *type* of risk.
* **Explaining the volatility.** A spike in `VOL_30D_AROUND_CROSS` with no news is probably flow or
  liquidity; the same spike with dense negative coverage is information being priced in — a different
  trade in each case.

**How I would actually implement it:** aggregate a per-ticker sentiment score over a window ending at
the cross date (a finance-tuned model such as FinBERT rather than a generic one, because "beat
expectations" or "outperform" are domain terms that general models misread), normalise it against the
stock's own history and its sector, and use it as a *filter* (take only Golden Crosses with
non-negative sentiment) or as a *ranking* variable when the screen returns more names than can be
traded. The proper test is whether the combined rule improves risk-adjusted returns out of sample
relative to the technical rule alone.

**Serious caveats:** news is noisy, largely already priced within minutes, and full of look-ahead
traps — the publication timestamp must be the one available in real time, not a revised one, and any
back-test must use point-in-time data. Sentiment scores are also often just a proxy for recent returns,
in which case they add nothing to a momentum signal. And most headlines about a stock *follow* the
price move ("Shares of X surge on…"), which creates a circularity that must be controlled for
explicitly.

### Critical thinking

#### From a trading perspective, is this strategy actionable on its own?

**As implemented here: no — it is a screening tool, not a trading strategy.** What the notebook
produces is a *watchlist*: "these names changed technical regime in the last two weeks." A tradeable
strategy needs several components this analysis deliberately does not have.

What is missing:

* **No exit rule, no position sizing, no risk budget.** Entry is only one of the four decisions a
  strategy must make (what, when, how much, when to get out); three are undefined here.
* **No costs, no capacity, no execution assumptions.** Commissions, spread and slippage are unmodelled,
  and entry at the signal close is not achievable in practice.
* **No statistical validation.** One year, one regime, overlapping events, survivorship-biased
  universe. The back-test above is an illustration of the *method*, not evidence of an edge; the
  t-tests are exactly there to show how weak the evidence is at this sample size.
* **No regime or fundamental context.** The rule is known to lose money in range-bound markets, and it
  has no mechanism to detect that it is in one.

What it *is* genuinely good for:

* **Systematic attention allocation.** Reducing 500 names to a handful of technically interesting ones
  is real value for a human analyst with limited time.
* **Risk management and discipline.** "Reduce exposure below the MA200" is a defensible, unemotional
  rule that has historically limited participation in deep drawdowns — its value is in avoiding the
  worst of bear markets, not in generating alpha.
* **A component in a multi-factor process**, where technicals govern timing and sizing while
  fundamentals or sentiment decide the underlying selection.

So: actionable as an input to a process, not as a standalone rule for capital allocation.

#### Based on the volatility observed post-Golden Cross, do these crosses consistently predict upward movement?

**No — "consistently" is the word that fails.** The event study computed above is the evidence: read
the `HIT_RATE`, `EXCESS_VS_BASELINE` and the p-values printed in the back-test section, and compare the
mean forward return after a Golden Cross with the unconditional baseline of the same universe over the
same horizons.

Three observations frame the interpretation, and they hold regardless of the exact numbers this run
produces:

1. **Any positive average is mostly beta, not signal.** Over 2024-05 → 2025-05 the index rose, so
   *any* long-only rule shows positive forward returns. The relevant number is the **excess over the
   baseline**, and the relevant question is whether it survives costs — which, at the magnitudes
   typically observed for this signal, it usually does not by much.
2. **The dispersion swamps the mean.** Even when the average forward return is positive, the hit rate
   sits close to a coin flip and the standard deviation of outcomes is an order of magnitude larger
   than the mean. Per-trade the signal is close to noise; whatever edge exists is a small shift of a
   very wide distribution, only extractable across many trades with disciplined sizing.
3. **Volatility conditions the outcome.** Crosses that print while local volatility is *below* the
   stock's 1-year level (calm, orderly trends) behave very differently from crosses that print inside a
   volatility spike, where the "trend" is often a violent bounce that mean-reverts. This is exactly why
   `VOL_30D_AROUND_CROSS` is carried alongside every signal, and it suggests the signal should be used
   *conditionally* rather than uniformly.

And a hard statistical caveat: with a few hundred overlapping events drawn from a single bullish year,
the confidence intervals are wide enough to contain zero for most horizons. The honest statement is
**"this sample cannot distinguish the post-Golden-Cross return from the baseline"**, not "Golden
Crosses do not work" — establishing either would require multiple decades and multiple regimes.

#### If you had to present this analysis to a portfolio manager, what conclusions would you emphasize? What caveats would you include?

**Conclusions (2 bullet points):**

* **We have an operational, reproducible screen over the entire index.** MA50/MA200 are computed for
  all ~500 constituents with a proper warm-up period, every crossover of the year is detected and
  stored in a tidy, filterable table, and the recent ones are enriched with volatility, sector and
  company context. This converts 500 price series into a short, ranked watchlist and a repeatable
  daily process — the deliverable is the *pipeline*, not a list of tickers.
* **The signal is a timing and risk-management overlay, not an alpha source on its own.** In the event
  study, forward returns after a Golden Cross are not reliably distinguishable from the unconditional
  return of the same universe once the market's own drift is taken into account, and the dispersion
  per trade is far larger than the average effect. The defensible use is to allocate attention, size
  positions and define unemotional exits — combined with fundamental or news-based selection (Section
  A) — rather than to trade the cross mechanically.

**Caveats (3 bullet points):**

* **Sample and survivorship limits.** One year of data, a single (bullish) regime, ~10 trading sessions
  in the "recent" window, and today's index membership — which excludes companies removed during the
  period and therefore biases results upward. Forward returns are also truncated for the most recent
  events. No conclusion here should be extrapolated to a different regime.
* **Frictions and implementability are not modelled.** No commissions, spread, slippage, taxes or
  capacity constraints; entry is assumed at the signal's close, which is not achievable in practice.
  A gross edge of a few tens of basis points can disappear entirely once these are included, and the
  strategy's turnover is highest exactly when markets are choppy and costs are highest.
* **The signal is lagging, crowded and endpoint-sensitive.** MA200 reacts with a ~100-day delay, so
  crosses confirm moves that are largely complete; the indicator is watched by everyone, so any edge is
  thin and decaying; and because the 14-day window is anchored to the last date of the data, the names
  on the list change with the run date. Whipsaws in range-bound markets are the dominant failure mode
  and require an explicit filter (minimum spread, confirmation days, regime filter) before any capital
  is committed.